# download_ui.ipynb
URLを貼り付けてボタンを押すだけでモデルをDLします。

**手順:** Cell 1 → Cell 2 を実行、あとはUIで操作

## Cell 1: 設定・トークン読み込み

In [ ]:
# ===== Cell 1: 設定・トークン読み込み =====
import os, json, subprocess, shutil, re
from pathlib import Path
from datetime import datetime

WORK_DIR  = "/workspace/runpod-slim"
MODEL_DIR = "/workspace/runpod-slim/ComfyUI/models"
LOG_FILE  = f"{WORK_DIR}/download_log.json"

ENV_FILE = f"{WORK_DIR}/.env"
def load_env():
    env = {}
    if os.path.exists(ENV_FILE):
        with open(ENV_FILE) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                env[k.strip()] = v.strip().strip('"').strip("'")
    return env

_env = load_env()
HF_TOKEN      = _env.get("HF_TOKEN", "")
CIVITAI_TOKEN = _env.get("CIVITAI_TOKEN", "")

try:
    import requests
except ImportError:
    subprocess.run(["pip", "install", "-q", "requests"], check=True, capture_output=True)
    import requests

def load_log():
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE) as f:
            return json.load(f)
    return {}

def save_log(log):
    with open(LOG_FILE, "w") as f:
        json.dump(log, f, ensure_ascii=False, indent=2)

CIVITAI_TYPE_MAP = {
    "LORA": "loras", "LoCon": "loras",
    "Checkpoint": "checkpoints", "VAE": "vae",
    "TextualInversion": "embeddings",
    "Upscaler": "upscale_models",
    "ControlNet": "controlnet",
}

def guess_folder(filename):
    fn = filename.lower()
    if fn.endswith(".gguf"):                                                          return "diffusion_models"
    if "ae.safetensors" in fn:                                                        return "vae"
    if any(x in fn for x in ["vae"]):                                                 return "vae"
    if any(x in fn for x in ["clip", "t5", "text_encoder"]):                          return "text_encoders"
    if any(x in fn for x in ["upscale", "esrgan", "realesrgan", "ultrasharp", "4x-", "2x-", "8x-"]): return "upscale_models"
    if any(x in fn for x in ["controlnet"]):                                          return "controlnet"
    if any(x in fn for x in ["lora", "locon"]):                                       return "loras"
    if fn.endswith(".pt") or fn.endswith(".pth"):                                      return "ultralytics/bbox"
    if fn.endswith(".safetensors"):                                                    return "diffusion_models"
    return "checkpoints"

def parse_civitai_url(url):
    for pat in [r"api/download/models/(\d+)", r"modelVersionId=(\d+)", r"/models/(\d+)"]:
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return None

def civitai_api(version_id):
    try:
        r = requests.get(f"https://civitai.com/api/v1/model-versions/{version_id}", timeout=10)
        return r.json() if r.status_code == 200 else None
    except Exception:
        return None

def _download(url, dest, label, token_header=None):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    tmp = dest + ".tmp"
    headers = token_header or {}
    try:
        with requests.get(url, headers=headers, stream=True, timeout=30) as r:
            r.raise_for_status()
            total = int(r.headers.get("Content-Length", 0))
            downloaded = 0
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=8*1024*1024):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        if total:
                            pct = downloaded / total * 100
                            print(f"  {downloaded/1e9:.2f}/{total/1e9:.2f} GB ({pct:.1f}%)", end="\r", flush=True)
        os.rename(tmp, dest)
        print(f"\n  ✅ {label}")
        return True
    except Exception as e:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise RuntimeError(str(e))

def process_url(url, dest_override=None):
    url = url.strip()
    if not url or url.startswith("#"):
        return None

    # HuggingFace
    if "huggingface.co" in url:
        filename = url.split("/")[-1].split("?")[0]
        folder   = dest_override.rstrip("/") if dest_override and dest_override != "-" else guess_folder(filename)
        dest     = f"{MODEL_DIR}/{folder}/{filename}"
        if os.path.exists(dest):
            return f"⏭️  スキップ（既存）: {filename}"
        headers = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
        print(f"⬇️  HF: {filename} → {folder}/")
        try:
            _download(url, dest, filename, headers)
            log = load_log()
            log[filename] = {"kind":"hf","url":url,"dest":dest,
                             "downloaded_at":datetime.now().isoformat()}
            save_log(log)
            return f"✅ {filename} → {folder}/"
        except RuntimeError as e:
            return f"❌ {filename}: {e}"

    # CivitAI
    if "civitai.com" in url or "civitai.red" in url:
        ver_id = parse_civitai_url(url)
        if not ver_id:
            return f"❌ バージョンID取得不可: {url}"
        api_data = civitai_api(ver_id)
        if api_data:
            files    = api_data.get("files", [])
            filename = files[0]["name"] if files else f"civitai_{ver_id}.safetensors"
            mtype    = api_data.get("model", {}).get("type", "")
            folder   = CIVITAI_TYPE_MAP.get(mtype) or guess_folder(filename)
        else:
            filename = f"civitai_{ver_id}.safetensors"
            folder   = "checkpoints"
            print(f"⚠️  API取得失敗 → {folder}/ に仮保存")
        dest = f"{MODEL_DIR}/{folder}/{filename}"
        if os.path.exists(dest):
            return f"⏭️  スキップ（既存）: {filename}"
        dl_url = f"https://civitai.com/api/download/models/{ver_id}"
        if CIVITAI_TOKEN:
            dl_url += f"?token={CIVITAI_TOKEN}"
        print(f"⬇️  CivitAI: {filename} → {folder}/")
        try:
            _download(dl_url, dest, filename)
            log = load_log()
            log[filename] = {"kind":"civitai","version_id":ver_id,"dest":dest,
                             "downloaded_at":datetime.now().isoformat()}
            save_log(log)
            return f"✅ {filename} → {folder}/"
        except RuntimeError as e:
            return f"❌ {filename}: {e}"

    # Ollama
    if not url.startswith("http"):
        if not shutil.which("ollama"):
            return f"⚠️ Ollama未インストール（setup の Cell 6.5 を先に実行）: {url}"
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
        if url in result.stdout:
            return f"⏭️  スキップ（既存）: {url}"
        print(f"⬇️  Ollama pull: {url}")
        r = subprocess.run(["ollama", "pull", url], capture_output=True, text=True)
        if r.returncode == 0:
            return f"✅ Ollama: {url}"
        else:
            return f"❌ Ollama pull 失敗: {url}"

    return f"⚠️  判定不能: {url}"

print("✅ Cell 1 完了")

## Cell 2: ダウンロードUI

In [ ]:
# ===== Cell 2: ダウンロードUI =====
import ipywidgets as widgets
from IPython.display import display, clear_output

LIST_FILE = f"{WORK_DIR}/download_list.txt"

def parse_list_urls():
    if not os.path.exists(LIST_FILE):
        return []
    entries = []
    with open(LIST_FILE) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                parts = [p.strip() for p in line.split("\t")]
                if len(parts) >= 3:
                    url = parts[2]
                    dest_override = parts[4] if len(parts) >= 5 and parts[4] != "-" else None
                    entries.append((url, dest_override))
    return entries

def append_to_list(urls_text):
    new_urls = [u.strip() for u in urls_text.splitlines() if u.strip() and not u.startswith("#")]
    if not new_urls:
        return
    existing = [e[0] for e in parse_list_urls()]
    with open(LIST_FILE, "a") as f:
        for url in new_urls:
            if url not in existing:
                name = url.split("/")[-1].split("?")[0].split(":")[0] or "model"
                kind = "hf" if "huggingface.co" in url else "civitai" if ("civitai.com" in url or "civitai.red" in url) else "ollama"
                f.write(f"{name}\t{kind}\t{url}\t-\t-\n")

def run_downloads(entries):
    if not entries:
        print("⚠️ URLがありません")
        return
    print(f"📋 {len(entries)} 件処理します\n")
    results = []
    for entry in entries:
        if isinstance(entry, tuple):
            url, dest_override = entry
        else:
            url, dest_override = entry, None
        result = process_url(url, dest_override)
        if result:
            results.append(result)
    print("\n━━━━━━━━━━━━━━━━━━━━")
    for r in results:
        print(f"  {r}")

# === UI パーツ ===
title = widgets.HTML("<h3 style='margin:0 0 12px;font-family:monospace'>📦 Model Downloader</h3>")

# ---- リストから実行 ----
btn_list = widgets.Button(
    description="リストから実行",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="36px"),
)
out_list = widgets.Output()

def on_list_click(b):
    with out_list:
        clear_output()
        entries = parse_list_urls()
        if not entries:
            print(f"⚠️ {LIST_FILE} にURLがありません")
            return
        run_downloads(entries)

btn_list.on_click(on_list_click)

list_section = widgets.VBox([
    widgets.HTML("<b style='font-family:monospace'>download_list.txt のURLを全件DL</b>"),
    btn_list,
    out_list,
])

# ---- URL追加してDL ----
url_input = widgets.Textarea(
    placeholder="URLを1行ずつ貼り付け\n\n例:\nhttps://civitai.com/models/123456/...\nhttps://huggingface.co/author/repo/resolve/main/model.safetensors\njaahas/qwen3.5-uncensored:9b",
    layout=widgets.Layout(width="100%", height="150px"),
)
btn_add = widgets.Button(
    description="追加してDL",
    button_style="success",
    layout=widgets.Layout(width="180px", height="36px"),
)
out_add = widgets.Output()

def on_add_click(b):
    with out_add:
        clear_output()
        text = url_input.value.strip()
        if not text:
            print("⚠️ URLを入力してください")
            return
        append_to_list(text)
        urls = [u.strip() for u in text.splitlines() if u.strip()]
        run_downloads(urls)
        url_input.value = ""

btn_add.on_click(on_add_click)

add_section = widgets.VBox([
    widgets.HTML("<b style='font-family:monospace'>URLを追加してDL（リストにも自動保存）</b>"),
    url_input,
    btn_add,
    out_add,
])

divider = widgets.HTML("<hr style='margin:16px 0;border-color:#333'>")

display(widgets.VBox([title, list_section, divider, add_section]))